# Task 2


###  Install libraries

In [2]:
!pip install scikit-learn gradio nltk

## Import libraries & download NLTK data

In [3]:
import nltk
import numpy as np
import gradio as gr
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Build your FAQ dataset

In [4]:
faq_data = {
    "What is Artificial Intelligence?": "Artificial Intelligence (AI) is the simulation of human intelligence in machines that are programmed to think, learn, and solve problems like humans.",
    "What is Machine Learning?": "Machine Learning is a subset of AI where systems learn from data and improve their performance over time without being explicitly programmed.",
    "What is Deep Learning?": "Deep Learning is a subset of Machine Learning that uses neural networks with many layers to analyze large amounts of data.",
    "What is a Neural Network?": "A Neural Network is a series of algorithms that mimic the human brain to recognize patterns and solve complex problems.",
    "What is Natural Language Processing?": "NLP is a branch of AI that helps computers understand, interpret, and generate human language.",
    "What is Computer Vision?": "Computer Vision is a field of AI that enables machines to interpret and understand visual information from images and videos.",
    "What is supervised learning?": "Supervised learning is a type of ML where the model is trained on labeled data — each input has a corresponding correct output.",
    "What is unsupervised learning?": "Unsupervised learning is where the model finds patterns in data without labeled responses.",
    "What is reinforcement learning?": "Reinforcement learning is a type of ML where an agent learns by interacting with its environment and receiving rewards or penalties.",
    "What is overfitting?": "Overfitting occurs when a model learns the training data too well, including noise, and performs poorly on new unseen data.",
    "What is a Large Language Model?": "A Large Language Model (LLM) is an AI model trained on massive amounts of text data to understand and generate human-like text. Examples include GPT and Claude.",
    "What is the difference between AI and ML?": "AI is the broad concept of machines being smart. ML is a specific technique used to achieve AI by training models on data.",
    "What are some real-world uses of AI?": "AI is used in healthcare (diagnosis), finance (fraud detection), self-driving cars, virtual assistants, recommendation systems, and much more.",
    "What is a chatbot?": "A chatbot is an AI-powered program that simulates conversation with users, typically to answer questions or provide support.",
    "What is the Turing Test?": "The Turing Test is a test proposed by Alan Turing to determine whether a machine can exhibit intelligent behavior indistinguishable from a human.",
}

questions = list(faq_data.keys())
answers = list(faq_data.values())

## Preprocess text with NLTK

In [5]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

cleaned_questions = [preprocess(q) for q in questions]

## Match user question using Cosine Similarity

In [6]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_questions)

def get_best_answer(user_input):
    if not user_input.strip():
        return "Please type a question."

    cleaned_input = preprocess(user_input)
    input_vector = vectorizer.transform([cleaned_input])

    similarities = cosine_similarity(input_vector, tfidf_matrix).flatten()
    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    if best_score < 0.1:
        return "Sorry, I couldn't find a matching answer. Try rephrasing your question."

    return f"**Q: {questions[best_index]}**\n\n{answers[best_index]}"

## Build the Chat UI with Gradio

In [7]:
def chat(user_message, history):
    response = get_best_answer(user_message)
    history.append((user_message, response))
    return "", history

with gr.Blocks(title="AI FAQ Chatbot") as app:
    gr.Markdown("# 🤖 AI FAQ Chatbot")
    gr.Markdown("Ask me anything about Artificial Intelligence!")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(placeholder="Type your question here...", label="Your Question")
    clear = gr.Button("Clear Chat")

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

app.launch()

/tmp/ipykernel_1915/2529725673.py:10: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_1915/2529725673.py:10: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d77a6ceec64e71a24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
